# Annexe - SVM avec noyaux (RBF, polynomial)

**Fichier complémentaire** au notebook principal `Groupe3_profils_atypiques_Sup.ipynb`.

Objectif : comparer empiriquement **LinearSVC** (retenu dans le projet) avec des **SVM à noyau** (`sklearn.svm.SVC`) pour illustrer pourquoi le noyau n'a pas été utilisé sur l'ensemble des ~514 000 lignes d'entraînement.

**Pipeline identique au projet :**
`8 features hors règles Excel` → `StandardScaler` → `ACP (5 comp.)` → SVM

---

### Limitation volontaire

| Modèle | Jeu d'entraînement | Raison |
|--------|-------------------|--------|
| **LinearSVC** | Train complet (~514 k) | Scalable |
| **SVC + noyau** | Sous-échantillon stratifié | Coût O(n²) — impraticable sur 500 k+ lignes |

Les SVM à noyau sont entraînés sur le **même sous-échantillon** pour une comparaison équitable entre noyaux.  
LinearSVC est aussi ré-entraîné sur ce sous-échantillon (ligne « LinearSVC (sous-éch.) ») en plus du train complet.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, RocCurveDisplay,
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = "users_labeled_manual.csv"
RANDOM_STATE = 42
KERNEL_SAMPLE_SIZE = 20_000  # ajuster si trop lent (ex. 10_000)

## 1. Chargement et features (identiques au notebook principal)

In [ ]:
features = [
    "followers_count", "friends_count",
    "avg_tweet_length", "avg_hashtags",
    "avg_favorites", "avg_retweet_count",
    "verified", "default_profile_image",
]

df = pd.read_csv(DATA_PATH)
df["verified"] = df["verified"].astype(int)
df["default_profile_image"] = df["default_profile_image"].astype(int)

X = df[features].copy()
y = df["label"].copy()

print(f"Dataset : {X.shape[0]:,} profils, {X.shape[1]} features")
print(y.value_counts(normalize=True).round(4) * 100)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=5, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Train : {X_train_pca.shape[0]:,} | Test : {X_test_pca.shape[0]:,}")
print(f"Variance ACP (5 comp.) : {pca.explained_variance_ratio_.sum() * 100:.2f} %")

## 2. Sous-échantillon stratifié pour les SVM à noyau

In [ ]:
n_sample = min(KERNEL_SAMPLE_SIZE, len(y_train))

splitter = StratifiedShuffleSplit(
    n_splits=1, train_size=n_sample, random_state=RANDOM_STATE
)
idx_sub, _ = next(splitter.split(X_train_pca, y_train))

X_sub = X_train_pca[idx_sub]
y_sub = y_train.iloc[idx_sub].values

print(f"Sous-échantillon : {len(y_sub):,} lignes")
print(pd.Series(y_sub).value_counts(normalize=True).round(4) * 100)

## 3. Entraînement et comparaison des variantes SVM

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    y_pred = model.predict(X_te)
    if hasattr(model, "decision_function"):
        y_score = model.decision_function(X_te)
    else:
        y_score = model.predict_proba(X_te)[:, 1]

    return {
        "Modèle": name,
        "Train (n)": len(y_tr),
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, pos_label=1),
        "Recall": recall_score(y_te, y_pred, pos_label=1),
        "F1": f1_score(y_te, y_pred, pos_label=1),
        "ROC-AUC": roc_auc_score(y_te, y_score),
        "Temps (s)": round(train_time, 1),
        "_pred": y_pred,
        "_score": y_score,
    }


models_config = [
    (
        "LinearSVC (train complet)",
        LinearSVC(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE),
        X_train_pca, y_train.values,
    ),
    (
        "LinearSVC (sous-éch.)",
        LinearSVC(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE),
        X_sub, y_sub,
    ),
    (
        "SVC RBF",
        SVC(kernel="rbf", class_weight="balanced", C=1.0, gamma="scale", random_state=RANDOM_STATE),
        X_sub, y_sub,
    ),
    (
        "SVC Polynomial (deg=2)",
        SVC(kernel="poly", degree=2, class_weight="balanced", C=1.0, random_state=RANDOM_STATE),
        X_sub, y_sub,
    ),
    (
        "SVC Polynomial (deg=3)",
        SVC(kernel="poly", degree=3, class_weight="balanced", C=1.0, random_state=RANDOM_STATE),
        X_sub, y_sub,
    ),
    (
        "SVC Sigmoid",
        SVC(kernel="sigmoid", class_weight="balanced", C=1.0, random_state=RANDOM_STATE),
        X_sub, y_sub,
    ),
]

results = []
fitted = {}

for name, model, X_tr, y_tr in models_config:
    print(f"Entraînement : {name}...")
    row = evaluate_model(name, model, X_tr, y_tr, X_test_pca, y_test.values)
    fitted[name] = (row.pop("_pred"), row.pop("_score"))
    results.append(row)
    print(f"  → F1 = {row['F1']:.3f} | Temps = {row['Temps (s)']} s\n")

results_df = pd.DataFrame(results).set_index("Modèle")
display(results_df.round(4))

## 4. Visualisations

In [ ]:
metrics = ["F1", "Recall", "Precision", "ROC-AUC"]
plot_df = results_df[metrics]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(metrics))
width = 0.12
colors = plt.cm.tab10(np.linspace(0, 0.9, len(plot_df)))

for i, (idx, row) in enumerate(plot_df.iterrows()):
    offset = (i - len(plot_df) / 2) * width + width / 2
    bars = ax.bar(x + offset, row.values, width, label=idx, color=colors[i], edgecolor="black")
    for bar, val in zip(bars, row.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title(f"Comparaison SVM — noyaux sur sous-échantillon ({n_sample:,} lignes)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC — modèles sur sous-échantillon uniquement
subsample_models = [n for n in fitted if n != "LinearSVC (train complet)"]
for name in subsample_models:
    _, score = fitted[name]
    RocCurveDisplay.from_predictions(y_test, score, name=name, ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title("Courbes ROC — évaluées sur le test complet")
axes[0].legend(fontsize=7, loc="lower right")

# Temps d'entraînement (échelle log)
times = results_df["Temps (s)"].sort_values()
axes[1].barh(times.index, times.values, color="#2196F3", edgecolor="black")
axes[1].set_xlabel("Temps (s)")
axes[1].set_title("Temps d'entraînement par variante")
axes[1].set_xscale("log")

plt.tight_layout()
plt.show()

## 5. (Optionnel) Grid search RBF sur le sous-échantillon

Recherche légère sur `C` et `gamma` — peut prendre plusieurs minutes.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", "auto", 0.01, 0.1],
}

grid = GridSearchCV(
    SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE),
    param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

print("Grid search RBF en cours...")
t0 = time.time()
grid.fit(X_sub, y_sub)
print(f"Durée : {time.time() - t0:.1f} s")
print(f"Meilleurs paramètres : {grid.best_params_}")
print(f"Meilleur F1 (CV) : {grid.best_score_:.3f}")

best_rbf = grid.best_estimator_
rbf_pred = best_rbf.predict(X_test_pca)
rbf_score = best_rbf.decision_function(X_test_pca)
print("\nRapport test — SVC RBF optimisé :")
print(classification_report(y_test, rbf_pred, target_names=["Normal", "Atypique"]))
print(f"F1 test : {f1_score(y_test, rbf_pred, pos_label=1):.3f}")
print(f"ROC-AUC test : {roc_auc_score(y_test, rbf_score):.3f}")

## 6. Synthèse et lecture pour la soutenance

### Ce que ce notebook montre

1. **LinearSVC sur le train complet** est le seul SVM réaliste à l'échelle du projet (~514 k lignes).
2. Les **SVM à noyau** (RBF, poly, sigmoid) ne peuvent être entraînés que sur un **sous-échantillon** (ici 20 000 lignes) : le temps et la mémoire explosent au-delà.
3. Sur le **même sous-échantillon**, on peut comparer linéaire vs noyaux : les noyaux peuvent légèrement améliorer le F1, mais **pas sur l'ensemble des données**.
4. Même un RBF optimisé par grid search reste limité au sous-échantillon — comparaison non équitable avec XGBoost entraîné sur le train complet.

### Phrase pour le jury

> « Nous avons testé les noyaux SVM dans un notebook annexe (`Groupe3_SVM_Noyaux_Annexe.ipynb`). Les résultats confirment que le noyau RBF peut améliorer légèrement le F1 sur un sous-échantillon, mais n'est pas applicable à nos 643 000 profils. D'où le choix de LinearSVC en baseline et de XGBoost pour la non-linéarité à pleine échelle. »

### Référence notebook principal

| Métrique | SVM linéaire (projet) | XGBoost (projet) |
|----------|----------------------|------------------|
| F1 | 0,361 | **0,443** |
| ROC-AUC | 0,670 | **0,794** |